# Lecture 14: 图像分割 (Image Segmentation)本笔记深入探讨图像分割：语义分割、FCN、U-Net架构、实例分割和mIoU度量。**学习目标：**- 理解语义分割与实例分割的区别- 掌握FCN和U-Net的核心架构- 实现转置卷积和跳跃连接- 理解mIoU等评价指标- 实现简单的分割模型**四步教学路径：** 直觉理解 -> 手动计算 -> 代码实现 -> 实验观察

## 目录1. 图像分割概述2. 语义分割 vs 实例分割3. 全卷积网络 (FCN)4. 转置卷积原理与实现5. U-Net架构详解6. 跳跃连接的作用7. mIoU度量与评价8. 实例分割概述 (Mask R-CNN)9. 作业与参考文献

## 1. 图像分割概述### 什么是图像分割？图像分割是对图像中每个像素进行分类的任务，目标是为每个像素分配一个类别标签。### 分割任务分类| 任务 | 输出 | 示例 ||---|---|---|| 语义分割 | 每像素一个类别 | 道路、建筑、天空 || 实例分割 | 每像素一个实例 | 车1、车2、人1 || 全景分割 | 语义+实例 | 背景+前景物体 |### 与分类的区别- **分类**：一张图 -> 一个标签- **目标检测**：一张图 -> 多个框+标签- **分割**：一张图 -> 每个像素一个标签（H x W 个预测）### 核心挑战1. **分辨率保持**：分类网络通过池化降低分辨率，分割需要恢复2. **空间信息**：分类丢失空间信息，分割需要保留3. **多尺度**：不同大小物体需要不同感受野

## 2. 语义分割 vs 实例分割### 语义分割- 不区分同类物体的不同实例- 输出：H x W x C 的概率图- 例如：所有"人"像素都是同一类别### 实例分割- 区分同类物体的不同实例- 输出：H x W x N 的mask，每个实例一个- 例如：第1个人、第2个人分别标记### 手动计算：输出尺寸假设输入32x32x3，3类语义分割：- 分类网络输出：1x1xC = 1x1x3（丢失空间信息）- 分割网络输出：32x32x3 = 3072个数（每像素一个预测）### 典型架构对比| 架构 | 类型 | 特点 ||---|---|---|| FCN | 语义 | 全卷积，端到端 || U-Net | 语义 | 编码-解码+跳跃连接 || DeepLab | 语义 | 空洞卷积+ASPP || Mask R-CNN | 实例 | 检测+分割 || Panoptic FPN | 全景 | 语义+实例融合 |

In [5]:
# -*- coding: utf-8 -*-import numpy as npimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltfrom matplotlib.patches import Rectangleimport matplotlib.patches as mpatchesnp.random.seed(42)# ============================================================# Create synthetic segmentation dataset# ============================================================def create_synthetic_image(size=64, n_objects=3):    # Create a synthetic image with shapes to segment    img = np.ones((size, size, 3)) * 0.3  # Background (gray)    mask = np.zeros((size, size), dtype=int)  # 0=background        colors = [[0.8, 0.2, 0.2], [0.2, 0.8, 0.2], [0.2, 0.2, 0.8], [0.8, 0.8, 0.2]]        for i in range(n_objects):        # Random object        shape_type = np.random.choice(['rect', 'circle'])        cx = np.random.randint(10, size - 10)        cy = np.random.randint(10, size - 10)        w = np.random.randint(8, 16)        h = np.random.randint(8, 16)        color = colors[i % len(colors)]                if shape_type == 'rect':            x1, x2 = max(0, cx - w//2), min(size, cx + w//2)            y1, y2 = max(0, cy - h//2), min(size, cy + h//2)            img[y1:y2, x1:x2] = color            mask[y1:y2, x1:x2] = i + 1        else:            yy, xx = np.ogrid[:size, :size]            circle = (yy - cy)**2 + (xx - cx)**2 <= (w//2)**2            img[circle] = color            mask[circle] = i + 1        return img, mask# Generate training datan_train = 50train_images = []train_masks = []for _ in range(n_train):    img, mask = create_synthetic_image(64)    train_images.append(img)    train_masks.append(mask)train_images = np.array(train_images)train_masks = np.array(train_masks)print(f"Training data: {train_images.shape}, masks: {train_masks.shape}")print(f"Unique mask values: {np.unique(train_masks)}")# Visualizefig, axes = plt.subplots(2, 3, figsize=(12, 8))for i in range(3):    axes[0, i].imshow(train_images[i])    axes[0, i].set_title(f'Image {i}', fontsize=12)    axes[0, i].axis('off')    axes[1, i].imshow(train_masks[i], cmap='nipy_spectral')    axes[1, i].set_title(f'Mask {i}', fontsize=12)    axes[1, i].axis('off')plt.suptitle('Synthetic Segmentation Dataset', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-14-segmentation/dataset.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 1 saved: dataset.png")

## 3. 全卷积网络 (FCN)### 直觉FCN将分类网络的全连接层替换为卷积层，实现像素级预测。核心思想：1. **编码器（下采样）**：用卷积+池化提取特征，降低空间分辨率2. **解码器（上采样）**：用转置卷积恢复空间分辨率3. **跳跃连接**：融合低层细节和高层语义### FCN架构FCN-32s: 直接从1/32分辨率上采样到原始尺寸（粗糙）FCN-16s: 融合1/16和1/32特征（中等）FCN-8s: 融合1/8、1/16和1/32特征（精细）### 手动计算：特征图尺寸变化输入: 64x64x3- Conv1+Pool: 32x32x64- Conv2+Pool: 16x16x128- Conv3+Pool: 8x8x256- Conv4+Pool: 4x4x512- Conv5+Pool: 2x2x512 (或 1x1x512)- 转置卷积 x5: 4->8->16->32->64- 输出: 64x64xC (C=类别数)每层下采样丢失3/4的空间信息！跳跃连接帮助恢复。

## 4. 转置卷积原理与实现### 直觉转置卷积（也叫"反卷积"）是上采样操作，将小特征图恢复为大特征图。### 手动计算**普通卷积**（stride=2, padding=0, kernel=3）：- 输入5x5 -> 输出2x2- 通过"im2col"展开为矩阵乘法**转置卷积**：- 输入2x2 -> 输出5x5- 本质是普通卷积的"转置"操作**具体步骤（kernel=3, stride=2, input=2x2）**：1. 在输入元素间插入stride-1个零 -> 3x32. 在四周padding kernel-1-padding个零 -> 7x73. 做普通卷积 -> 5x5**数值示例：**输入 = [[1, 2], [3, 4]]kernel = [[1, 0, 1], [0, 1, 0], [1, 0, 1]]插入零后：[[1, 0, 2], [0, 0, 0], [3, 0, 4]]padding后(7x7) -> 卷积 -> 输出5x5### 双线性插值初始化转置卷积的权重可以用双线性插值公式初始化，使上采样更平滑。

In [8]:
# ============================================================# Transposed Convolution Implementation# ============================================================def conv2d_numpy(x, w, stride=1, padding=0):    # 2D convolution (cross-correlation)    if padding > 0:        x = np.pad(x, ((padding, padding), (padding, padding)), mode='constant')    H, W = x.shape    KH, KW = w.shape    OH = (H - KH) // stride + 1    OW = (W - KW) // stride + 1    out = np.zeros((OH, OW))    for i in range(OH):        for j in range(OW):            out[i, j] = np.sum(x[i*stride:i*stride+KH, j*stride:j*stride+KW] * w)    return outdef transposed_conv2d(x, w, stride=2, padding=0):    # Transposed convolution (fractionally-strided convolution)    H_in, W_in = x.shape    KH, KW = w.shape    # Output size    H_out = (H_in - 1) * stride + KH - 2 * padding    W_out = (W_in - 1) * stride + KW - 2 * padding        # Insert zeros between input elements    x_expanded = np.zeros((H_out + KH - 1, W_out + KW - 1))    for i in range(H_in):        for j in range(W_in):            x_expanded[i * stride, j * stride] = x[i, j]        # Apply convolution with flipped kernel    out = np.zeros((H_out, W_out))    for i in range(H_out):        for j in range(W_out):            out[i, j] = np.sum(x_expanded[i:i+KH, j:j+KW] * w)        return out# Demonstrate transposed convolutionx_small = np.array([[1, 2], [3, 4]], dtype=float)kernel = np.array([[1, 0, 1], [0, 1, 0], [1, 0, 1]], dtype=float)result = transposed_conv2d(x_small, kernel, stride=2, padding=0)print("=== Transposed Convolution ===")print(f"Input shape: {x_small.shape}")print(f"Kernel shape: {kernel.shape}")print(f"Output shape: {result.shape}")print(f"Output:\n{result}")# Visualizefig, axes = plt.subplots(1, 3, figsize=(12, 4))axes[0].imshow(x_small, cmap='viridis')axes[0].set_title(f'Input ({x_small.shape[0]}x{x_small.shape[1]})', fontsize=12)for i in range(x_small.shape[0]):    for j in range(x_small.shape[1]):        axes[0].text(j, i, f'{x_small[i,j]:.0f}', ha='center', va='center', color='white', fontsize=14)axes[0].axis('off')axes[1].imshow(kernel, cmap='viridis')axes[1].set_title(f'Kernel ({kernel.shape[0]}x{kernel.shape[1]})', fontsize=12)axes[1].axis('off')axes[2].imshow(result, cmap='viridis')axes[2].set_title(f'Output ({result.shape[0]}x{result.shape[1]})', fontsize=12)axes[2].axis('off')plt.suptitle('Transposed Convolution Demo', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-14-segmentation/transposed_conv.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 2 saved: transposed_conv.png")

## 5. U-Net架构详解### 直觉U-Net因其U形结构得名。左半部分是编码器（下采样路径），右半部分是解码器（上采样路径），中间有跳跃连接将编码器特征传递给解码器。### U-Net核心组件1. **编码器**：每次2个3x3卷积 + ReLU + 2x2最大池化2. **瓶颈层**：2个3x3卷积 + ReLU（最低分辨率）3. **解码器**：转置卷积 + 拼接跳跃连接 + 2个3x3卷积 + ReLU4. **输出层**：1x1卷积映射到类别数### 手动计算：U-Net尺寸变化（输入64x64）| 层 | 操作 | 尺寸 | 通道 ||---|---|---|---|| 编码1 | conv+conv+pool | 64->32 | 64 || 编码2 | conv+conv+pool | 32->16 | 128 || 编码3 | conv+conv+pool | 16->8 | 256 || 编码4 | conv+conv+pool | 8->4 | 512 || 瓶颈 | conv+conv | 4 | 1024 || 解码1 | upconv+concat | 4->8 | 512+512=1024 || 解码2 | upconv+concat | 8->16 | 256+256=512 || 解码3 | upconv+concat | 16->32 | 128+128=256 || 解码4 | upconv+concat | 32->64 | 64+64=128 || 输出 | 1x1 conv | 64 | C |### U-Net的优势1. **跳跃连接**：融合高层语义和低层细节2. **对称结构**：解码器与编码器对称，逐步恢复分辨率3. **少量数据**：在医学图像（数据少）上效果特别好

In [10]:
# ============================================================# Simplified U-Net Implementation# ============================================================class SimpleUNet:    # Simplified U-Net for 2D segmentation    def __init__(self, in_channels=3, n_classes=4, base_filters=16):        np.random.seed(42)        self.in_ch = in_channels        self.n_classes = n_classes        self.f = base_filters                # Encoder weights        self.enc1_w = [np.random.randn(3, 3, in_channels, self.f) * 0.01,                       np.random.randn(3, 3, self.f, self.f) * 0.01]        self.enc2_w = [np.random.randn(3, 3, self.f, self.f * 2) * 0.01,                       np.random.randn(3, 3, self.f * 2, self.f * 2) * 0.01]        self.enc3_w = [np.random.randn(3, 3, self.f * 2, self.f * 4) * 0.01,                       np.random.randn(3, 3, self.f * 4, self.f * 4) * 0.01]                # Bottleneck        self.bot_w = [np.random.randn(3, 3, self.f * 4, self.f * 8) * 0.01,                      np.random.randn(3, 3, self.f * 8, self.f * 8) * 0.01]                # Decoder (transposed conv + 2 convs)        self.dec3_w = [np.random.randn(2, 2, self.f * 8, self.f * 4) * 0.01,  # upconv                       np.random.randn(3, 3, self.f * 8, self.f * 4) * 0.01,  # conv1                       np.random.randn(3, 3, self.f * 4, self.f * 4) * 0.01]  # conv2        self.dec2_w = [np.random.randn(2, 2, self.f * 4, self.f * 2) * 0.01,                       np.random.randn(3, 3, self.f * 4, self.f * 2) * 0.01,                       np.random.randn(3, 3, self.f * 2, self.f * 2) * 0.01]        self.dec1_w = [np.random.randn(2, 2, self.f * 2, self.f) * 0.01,                       np.random.randn(3, 3, self.f * 2, self.f) * 0.01,                       np.random.randn(3, 3, self.f, self.f) * 0.01]                # Output        self.out_w = np.random.randn(1, 1, self.f, n_classes) * 0.01                # Storage for skip connections        self.skip_features = {}        def _conv_block(self, x, weights):        # Two conv + relu layers        for w in weights:            # Simple convolution (stride=1, pad=1)            H, W, C = x.shape            x_padded = np.pad(x, ((1,1),(1,1),(0,0)), mode='constant')            out = np.zeros((H, W, w.shape[3]))            for c_out in range(w.shape[3]):                for i in range(H):                    for j in range(W):                        out[i, j, c_out] = np.sum(x_padded[i:i+3, j:j+3, :] * w[:, :, :, c_out])            x = np.maximum(0, out)        return x        def _maxpool(self, x, size=2):        H, W, C = x.shape        OH, OW = H // size, W // size        out = np.zeros((OH, OW, C))        for c in range(C):            for i in range(OH):                for j in range(OW):                    out[i, j, c] = np.max(x[i*size:i*size+size, j*size:j*size+size, c])        return out        def _upsample(self, x, factor=2):        # Simple nearest-neighbor upsampling        H, W, C = x.shape        out = np.repeat(np.repeat(x, factor, axis=0), factor, axis=1)        return out        def forward(self, x):        # Encoder        e1 = self._conv_block(x, self.enc1_w)        self.skip_features['e1'] = e1        p1 = self._maxpool(e1)                e2 = self._conv_block(p1, self.enc2_w)        self.skip_features['e2'] = e2        p2 = self._maxpool(e2)                e3 = self._conv_block(p2, self.enc3_w)        self.skip_features['e3'] = e3        p3 = self._maxpool(e3)                # Bottleneck        bot = self._conv_block(p3, self.bot_w)                # Decoder with skip connections        d3 = self._upsample(bot)        # Crop skip if needed        d3 = self._conv_block(np.concatenate([d3, self.skip_features['e3']], axis=-1), [self.dec3_w[1], self.dec3_w[2]])                d2 = self._upsample(d3)        d2 = self._conv_block(np.concatenate([d2, self.skip_features['e2']], axis=-1), [self.dec2_w[1], self.dec2_w[2]])                d1 = self._upsample(d2)        d1 = self._conv_block(np.concatenate([d1, self.skip_features['e1']], axis=-1), [self.dec1_w[1], self.dec1_w[2]])                # Output        out = np.zeros((d1.shape[0], d1.shape[1], self.n_classes))        for c in range(self.n_classes):            out[:, :, c] = np.sum(d1 * self.out_w[:, :, :, c], axis=-1)                return outprint("SimpleUNet defined.")print("Architecture: 3-level encoder + bottleneck + 3-level decoder")

In [11]:
# ============================================================# Test U-Net on synthetic data# ============================================================model = SimpleUNet(in_channels=3, n_classes=4, base_filters=8)# Forward pass on one imagetest_img = train_images[0]  # 64x64x3output = model.forward(test_img)pred_mask = output.argmax(axis=-1)print(f"Input shape: {test_img.shape}")print(f"Output shape: {output.shape}")print(f"Predicted mask shape: {pred_mask.shape}")print(f"Ground truth mask shape: {train_masks[0].shape}")fig, axes = plt.subplots(1, 3, figsize=(15, 5))axes[0].imshow(test_img)axes[0].set_title('Input Image', fontsize=12)axes[0].axis('off')axes[1].imshow(train_masks[0], cmap='nipy_spectral')axes[1].set_title('Ground Truth Mask', fontsize=12)axes[1].axis('off')axes[2].imshow(pred_mask, cmap='nipy_spectral')axes[2].set_title('Predicted Mask (untrained)', fontsize=12)axes[2].axis('off')plt.suptitle('U-Net Segmentation (Untrained)', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-14-segmentation/unet_output.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 3 saved: unet_output.png")

## 6. 跳跃连接的作用### 直觉跳跃连接将编码器的中间特征直接传递给解码器，解决了两个问题：1. **恢复空间细节**：编码器在池化时丢失了细节，跳跃连接补回2. **梯度流动**：帮助梯度更容易回传到浅层### 手动分析没有跳跃连接：- 解码器只有瓶颈层的信息（如4x4）- 需要从4x4恢复到64x64 -> 信息严重不足有跳跃连接：- 解码器在8x8时获得编码器8x8的细节- 在16x16时获得编码器16x16的细节- 逐步融合，信息更丰富### 不同跳跃策略| 策略 | 做法 | 效果 ||---|---|---|| 拼接 (U-Net) | 直接拼接特征通道 | 信息最丰富 || 相加 (FCN) | 对应元素相加 | 参数少 || 门控 | 学习权重控制信息流 | 更灵活 |

In [13]:
# ============================================================# Visualize skip connections effect# ============================================================# Simulate with and without skip connectionsnp.random.seed(42)# Low-resolution feature (bottleneck)bottleneck = np.random.randn(8, 8, 32)# Upsampled without skip (just interpolation)upsampled_no_skip = np.repeat(np.repeat(bottleneck.mean(-1), 8, axis=0), 8, axis=1)upsampled_no_skip = upsampled_no_skip + np.random.randn(64, 64) * 0.1# High-res skip featureskip_feat = train_masks[0].astype(float)# With skip: combine upsampled + skipupsampled_with_skip = upsampled_no_skip * 0.5 + skip_feat * 0.5fig, axes = plt.subplots(1, 3, figsize=(15, 5))axes[0].imshow(upsampled_no_skip, cmap='viridis')axes[0].set_title('Without Skip\n(only bottleneck info)', fontsize=12)axes[0].axis('off')axes[1].imshow(upsampled_with_skip, cmap='viridis')axes[1].set_title('With Skip\n(bottleneck + detail)', fontsize=12)axes[1].axis('off')axes[2].imshow(train_masks[0], cmap='nipy_spectral')axes[2].set_title('Ground Truth', fontsize=12)axes[2].axis('off')plt.suptitle('Effect of Skip Connections', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-14-segmentation/skip_effect.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 4 saved: skip_effect.png")

## 7. mIoU度量与评价### 直觉mIoU (mean Intersection over Union) 是分割任务的标准评价指标。### 手动计算对于每个类别c：IoU_c = TP_c / (TP_c + FP_c + FN_c)其中：- TP (True Positive)：正确预测为c的像素数- FP (False Positive)：错误预测为c的像素数- FN (False Negative)：应为c但预测为其他类的像素数mIoU = (1/C) * sum(IoU_c)**数值示例：**类别1：TP=100, FP=20, FN=10 -> IoU = 100/130 = 0.769类别2：TP=200, FP=50, FN=30 -> IoU = 200/280 = 0.714类别3：TP=50, FP=10, FN=5 -> IoU = 50/65 = 0.769mIoU = (0.769 + 0.714 + 0.769) / 3 = 0.751### 其他指标- Pixel Accuracy: 正确像素 / 总像素- Mean Accuracy: 各类别精度的平均- Frequency Weighted IoU: 按频率加权的IoU

In [15]:
# ============================================================# Segmentation Metrics Implementation# ============================================================def compute_iou(pred, gt, n_classes):    # Compute IoU for each class    iou = np.zeros(n_classes)    for c in range(n_classes):        pred_c = pred == c        gt_c = gt == c        intersection = np.logical_and(pred_c, gt_c).sum()        union = np.logical_or(pred_c, gt_c).sum()        if union > 0:            iou[c] = intersection / union        else:            iou[c] = float('nan')  # Class not present    return ioudef compute_miou(pred, gt, n_classes):    # Compute mean IoU    ious = compute_iou(pred, gt, n_classes)    valid = ious[~np.isnan(ious)]    if len(valid) > 0:        return np.mean(valid), ious    return 0.0, iousdef pixel_accuracy(pred, gt):    # Compute pixel accuracy    return (pred == gt).mean()# Test metrics on our predictionmiou, per_class_iou = compute_miou(pred_mask, train_masks[0], 4)pa = pixel_accuracy(pred_mask, train_masks[0])print("=== Segmentation Metrics ===")print(f"Pixel Accuracy: {pa:.4f}")print(f"mIoU: {miou:.4f}")for c in range(4):    print(f"  Class {c} IoU: {per_class_iou[c]:.4f}")# Evaluate across multiple samplesall_mious = []all_pas = []for i in range(min(10, len(train_images))):    out = model.forward(train_images[i])    pred = out.argmax(-1)    m, _ = compute_miou(pred, train_masks[i], 4)    p = pixel_accuracy(pred, train_masks[i])    all_mious.append(m)    all_pas.append(p)print(f"\nAverage over 10 samples:")print(f"  Mean mIoU: {np.mean(all_mious):.4f}")print(f"  Mean Pixel Accuracy: {np.mean(all_pas):.4f}")

In [16]:
# ============================================================# Visualize IoU computation# ============================================================# Show IoU computation for one classfig, axes = plt.subplots(1, 4, figsize=(16, 4))# Ground truth for class 1gt_class1 = (train_masks[0] == 1)pred_class1 = (pred_mask == 1)axes[0].imshow(gt_class1, cmap='Blues')axes[0].set_title('Ground Truth\nClass 1', fontsize=11)axes[0].axis('off')axes[1].imshow(pred_class1, cmap='Reds')axes[1].set_title('Prediction\nClass 1', fontsize=11)axes[1].axis('off')# Intersectionintersection = np.logical_and(gt_class1, pred_class1)axes[2].imshow(intersection, cmap='Greens')axes[2].set_title(f'Intersection\n({intersection.sum()} pixels)', fontsize=11)axes[2].axis('off')# Unionunion = np.logical_or(gt_class1, pred_class1)axes[3].imshow(union, cmap='Oranges')axes[3].set_title(f'Union\n({union.sum()} pixels)', fontsize=11)axes[3].axis('off')iou_val = intersection.sum() / max(union.sum(), 1)plt.suptitle(f'IoU Computation for Class 1: IoU = {iou_val:.4f}', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-14-segmentation/iou_viz.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 5 saved: iou_viz.png")print(f"IoU for class 1: {iou_val:.4f}")

## 8. 空洞卷积 (Dilated/Atrous Convolution)### 直觉空洞卷积在卷积核元素间插入"空洞"，在不增加参数量的情况下扩大感受野。### 手动计算普通3x3卷积，dilation=1：感受野=3x3空洞3x3卷积，dilation=2：感受野=5x5空洞3x3卷积，dilation=4：感受野=9x9**为什么对分割有用？**分割需要大感受野（看到整个物体）但保持高分辨率。空洞卷积在不降低分辨率的情况下扩大感受野。### DeepLab的ASPPASPP (Atrous Spatial Pyramid Pooling) 使用不同空洞率的卷积并行处理，捕获多尺度信息。

In [18]:
# ============================================================# Dilated convolution visualization# ============================================================def dilated_conv2d(x, w, dilation=1):    # 2D dilated (atrous) convolution    H, W = x.shape    KH, KW = w.shape    # Effective kernel size    EKH = (KH - 1) * dilation + 1    EKW = (KW - 1) * dilation + 1    OH = H - EKH + 1    OW = W - EKW + 1    out = np.zeros((OH, OW))    for i in range(OH):        for j in range(OW):            for ki in range(KH):                for kj in range(KW):                    out[i, j] += x[i + ki * dilation, j + kj * dilation] * w[ki, kj]    return out# Visualize receptive fieldfig, axes = plt.subplots(1, 3, figsize=(12, 4))for idx, dilation in enumerate([1, 2, 4]):    ax = axes[idx]    kernel = np.ones((3, 3))    # Create a sample image    img_test = np.zeros((11, 11))    img_test[5, 5] = 1    # Mark receptive field of center output pixel    receptive = np.zeros((11, 11))    for ki in range(3):        for kj in range(3):            r = 5 - 1 + ki * dilation            c = 5 - 1 + kj * dilation            if 0 <= r < 11 and 0 <= c < 11:                receptive[r, c] = 1        ax.imshow(receptive, cmap='Blues', vmin=0, vmax=1)    eff_size = (3 - 1) * dilation + 1    ax.set_title(f'Dilation={dilation}\nReceptive field={eff_size}x{eff_size}', fontsize=11)    ax.axis('off')plt.suptitle('Dilated Convolution: Receptive Field', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-14-segmentation/dilated.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 6 saved: dilated.png")

## 9. 实例分割概述 (Mask R-CNN)### 直觉实例分割结合了目标检测和语义分割：先检测每个物体实例，再在每个实例区域内做分割。### Mask R-CNN流程1. **特征提取**：用CNN骨干提取特征2. **RPN**：Region Proposal Network生成候选框3. **ROI Align**：精确对齐候选区域特征4. **分类+回归+Mask**：三个并行分支   - 分类：预测类别   - 回归：精修边界框   - Mask：在ROI内做像素级分割### 与语义分割的区别| 方面 | 语义分割 | 实例分割 ||---|---|---|| 输出 | H x W x C | N个mask || 实例区分 | 不区分 | 区分 || 框架 | FCN/U-Net | Mask R-CNN || 速度 | 较快 | 较慢 || 复杂度 | 较低 | 较高 |### 手动计算：Mask预测对于每个ROI（如14x14区域）：- 分类分支输出C+1个类别概率- Mask分支输出C+1个14x14的mask- 最终取预测类别的mask，二值化得到实例分割

In [20]:
# ============================================================# Instance segmentation simulation# ============================================================np.random.seed(42)# Simulate instance segmentation resultsfig, axes = plt.subplots(1, 3, figsize=(15, 5))# Original imageimg, gt_mask = create_synthetic_image(64, n_objects=4)axes[0].imshow(img)axes[0].set_title('Input Image', fontsize=12)axes[0].axis('off')# Semantic segmentation (all objects same class)sem_mask = (gt_mask > 0).astype(int)axes[1].imshow(sem_mask, cmap='Blues')axes[1].set_title('Semantic Segmentation\n(all objects = 1 class)', fontsize=11)axes[1].axis('off')# Instance segmentation (each object unique)inst_mask = gt_mask.copy()axes[2].imshow(inst_mask, cmap='nipy_spectral')axes[2].set_title('Instance Segmentation\n(each object unique)', fontsize=11)axes[2].axis('off')plt.suptitle('Semantic vs Instance Segmentation', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-14-segmentation/semantic_vs_instance.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 7 saved: semantic_vs_instance.png")# Count instancesn_instances = len(np.unique(gt_mask)) - 1  # Exclude backgroundprint(f"Number of instances: {n_instances}")print(f"Semantic classes: 2 (background + foreground)")print(f"Instance IDs: {list(np.unique(gt_mask))}")

## 10. 后处理：CRF与形态学操作### 直觉分割结果常需后处理提升质量：1. **CRF (Conditional Random Field)**：利用像素间关系平滑边界2. **形态学操作**：开运算（去噪点）、闭运算（填孔洞）3. **超像素聚合**：将相似超像素合并### 手动计算：形态学操作**腐蚀 (Erosion)**：对每个像素，如果邻域内有背景则变为背景**膨胀 (Dilation)**：对每个像素，如果邻域内有前景则变为前景**开运算**：先腐蚀后膨胀（去除小噪点）**闭运算**：先膨胀后腐蚀（填充小孔洞）**数值示例（3x3核）：**mask = [[0,0,0,0,0],        [0,1,1,0,0],        [0,1,1,1,0],        [0,0,1,1,0],        [0,0,0,0,0]]腐蚀后（去掉边缘1格）:        [[0,0,0,0,0],         [0,0,1,0,0],         [0,0,1,1,0],         [0,0,0,1,0],         [0,0,0,0,0]]

In [22]:
# ============================================================# Morphological operations# ============================================================def erode(mask, kernel_size=3):    # Erosion    from scipy.ndimage import minimum_filter    # Simplified: use binary erosion    H, W = mask.shape    k = kernel_size // 2    out = np.zeros_like(mask)    for i in range(H):        for j in range(W):            region = mask[max(0,i-k):i+k+1, max(0,j-k):j+k+1]            out[i, j] = region.min()    return outdef dilate(mask, kernel_size=3):    # Dilation    H, W = mask.shape    k = kernel_size // 2    out = np.zeros_like(mask)    for i in range(H):        for j in range(W):            region = mask[max(0,i-k):i+k+1, max(0,j-k):j+k+1]            out[i, j] = region.max()    return outdef opening(mask, kernel_size=3):    # Opening = erode then dilate    return dilate(erode(mask, kernel_size), kernel_size)def closing(mask, kernel_size=3):    # Closing = dilate then erode    return erode(dilate(mask, kernel_size), kernel_size)# Test morphological operationsbinary_mask = (train_masks[0] > 0).astype(int)# Add some noisenoisy_mask = binary_mask.copy()noisy_mask[5:8, 5:8] = 1  # Small noise blobnoisy_mask[30:32, 30:32] = 0  # Small holeopened = opening(noisy_mask)closed = closing(noisy_mask)both = closing(opening(noisy_mask))fig, axes = plt.subplots(1, 4, figsize=(16, 4))axes[0].imshow(noisy_mask, cmap='gray')axes[0].set_title('Noisy Mask\n(noise + holes)', fontsize=11)axes[0].axis('off')axes[1].imshow(opened, cmap='gray')axes[1].set_title('Opening\n(removed noise)', fontsize=11)axes[1].axis('off')axes[2].imshow(closed, cmap='gray')axes[2].set_title('Closing\n(filled holes)', fontsize=11)axes[2].axis('off')axes[3].imshow(both, cmap='gray')axes[3].set_title('Open + Close\n(cleaned)', fontsize=11)axes[3].axis('off')plt.suptitle('Morphological Operations', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-14-segmentation/morphology.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 8 saved: morphology.png")

## 11. 分割架构对比### 主流架构总结| 架构 | 年份 | 核心创新 | mIoU(ImageNet) ||---|---|---|---|| FCN | 2015 | 全卷积+跳跃 | ~50% || U-Net | 2015 | 对称编解码+密集跳跃 | ~50% || DeepLab v2 | 2016 | 空洞卷积+ASPP | ~60% || DeepLab v3 | 2017 | 改进ASPP+BN | ~67% || DeepLab v3+ | 2018 | 编解码+ASPP | ~80% || PSPNet | 2017 | 金字塔池化 | ~66% || SegFormer | 2021 | Transformer | ~84% |### 选择建议- 医学图像（数据少）-> U-Net- 通用语义分割 -> DeepLab v3+- 追求精度 -> SegFormer/Mask2Former- 实例分割 -> Mask R-CNN / YOLACT- 实时分割 -> BiSeNet / Fast-SCNN

In [24]:
# ============================================================# U-Net architecture diagram# ============================================================fig, ax = plt.subplots(figsize=(14, 8))ax.set_xlim(-1, 15)ax.set_ylim(-1, 8)ax.axis('off')# Encoder boxesenc_y = [6, 5, 4, 3]enc_x = [0, 2, 4, 6]enc_labels = ['64x64\n64ch', '32x32\n128ch', '16x16\n256ch', '8x8\n512ch']enc_colors = ['#FFE0B2', '#FFCC80', '#FFB74D', '#FFA726']# Decoder boxesdec_x = [8, 10, 12, 14]dec_labels = ['8x8\n1024ch', '16x16\n512ch', '32x32\n256ch', '64x64\n128ch']dec_colors = ['#B3E5FC', '#81D4FA', '#4FC3F7', '#29B6F6']# Bottleneckax.add_patch(plt.Rectangle((7, 2.5), 0.8, 1, facecolor='#E91E63', edgecolor='black'))ax.text(7.4, 3, 'Bot\n4x4', ha='center', va='center', fontsize=8, color='white', fontweight='bold')# Draw encoderfor i, (x, y, label, color) in enumerate(zip(enc_x, enc_y, enc_labels, enc_colors)):    ax.add_patch(plt.Rectangle((x, y - 0.4), 1.5, 0.8, facecolor=color, edgecolor='black'))    ax.text(x + 0.75, y, label, ha='center', va='center', fontsize=8)    if i < 3:        ax.annotate('', xy=(x + 2, y - 1), xytext=(x + 1.5, y - 0.4),                    arrowprops=dict(arrowstyle='->', color='gray'))# Draw decoderfor i, (x, y, label, color) in enumerate(zip(dec_x, enc_y, dec_labels, dec_colors)):    ax.add_patch(plt.Rectangle((x, y - 0.4), 1.5, 0.8, facecolor=color, edgecolor='black'))    ax.text(x + 0.75, y, label, ha='center', va='center', fontsize=8)    if i < 3:        ax.annotate('', xy=(x + 2, y - 1), xytext=(x + 1.5, y - 0.4),                    arrowprops=dict(arrowstyle='->', color='gray'))# Skip connections (dashed arrows)for i in range(3):    ax.annotate('', xy=(dec_x[i] + 0.75, enc_y[i] - 0.4),                xytext=(enc_x[i] + 1.5, enc_y[i] - 0.4),                arrowprops=dict(arrowstyle='->', color='red', linestyle='--', lw=1.5))    ax.text((enc_x[i] + dec_x[i]) / 2 + 0.75, enc_y[i] + 0.6, 'skip',             ha='center', fontsize=7, color='red')ax.set_title('U-Net Architecture', fontsize=16, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-14-segmentation/unet_arch.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 9 saved: unet_arch.png")

## 11.5 分割损失函数### 直觉分割任务不仅可以用交叉熵损失，还有更专门的损失函数。### 1. 交叉熵损失 (CE)L_ce = -(1/N) * sum(y * log(p) + (1-y) * log(1-p))缺点：类别不平衡时表现差（如90%背景，10%前景）### 2. Dice LossL_dice = 1 - (2 * |pred cap gt|) / (|pred| + |gt|)优点：直接优化IoU，对类别不平衡不敏感### 3. Focal LossL_focal = -alpha * (1 - p)^gamma * log(p)优点：降低易分样本的权重，让模型关注难分样本### 手动计算Dicepred = [1, 1, 0, 1, 0], gt = [1, 0, 0, 1, 1]- 交集 |pred cap gt| = |[1, 0, 0, 1, 0]| = 2- |pred| = 3, |gt| = 3- Dice = 2*2 / (3+3) = 4/6 = 0.667- Dice Loss = 1 - 0.667 = 0.333

In [26]:
# ============================================================# Loss functions for segmentation# ============================================================def cross_entropy_loss(pred_logits, gt_mask, n_classes):    # Standard cross-entropy loss    H, W = gt_mask.shape    # Softmax    shifted = pred_logits - pred_logits.max(axis=-1, keepdims=True)    exp = np.exp(shifted)    probs = exp / exp.sum(axis=-1, keepdims=True)        loss = 0    for c in range(n_classes):        gt_c = (gt_mask == c).astype(float)        pred_c = probs[:, :, c]        loss -= np.sum(gt_c * np.log(pred_c + 1e-12))    return loss / (H * W)def dice_loss(pred_mask, gt_mask, n_classes):    # Multi-class Dice loss    total = 0    for c in range(n_classes):        pred_c = (pred_mask == c).astype(float)        gt_c = (gt_mask == c).astype(float)        intersection = np.sum(pred_c * gt_c)        union = np.sum(pred_c) + np.sum(gt_c)        if union > 0:            total += 1 - 2 * intersection / union        else:            total += 0  # Class not present    return total / n_classesdef focal_loss(pred_logits, gt_mask, n_classes, alpha=0.25, gamma=2.0):    # Focal loss    shifted = pred_logits - pred_logits.max(axis=-1, keepdims=True)    exp = np.exp(shifted)    probs = exp / exp.sum(axis=-1, keepdims=True)        H, W = gt_mask.shape    loss = 0    for c in range(n_classes):        gt_c = (gt_mask == c).astype(float)        pred_c = probs[:, :, c]        loss += np.sum(alpha * (1 - pred_c) ** gamma * gt_c * (-np.log(pred_c + 1e-12)))    return loss / (H * W)# Compare lossesprint("=== Loss Function Comparison ===")# Simulate predictions with different qualitynp.random.seed(42)pred_good = train_masks[0].copy()  # Perfect predictionpred_bad = np.random.randint(0, 4, (64, 64))  # Random predictionpred_partial = train_masks[0].copy()# Corrupt 30% of predictionsnoise_mask = np.random.rand(64, 64) < 0.3pred_partial[noise_mask] = np.random.randint(0, 4, noise_mask.sum())for name, pred in [('Perfect', pred_good), ('30% Noise', pred_partial), ('Random', pred_bad)]:    ce = cross_entropy_loss(np.eye(4)[pred], train_masks[0], 4)    dl = dice_loss(pred, train_masks[0], 4)    print(f"{name}: Dice Loss={dl:.4f}")

In [27]:
# ============================================================# Visualize loss landscape comparison# ============================================================# Simulate training with different lossesnp.random.seed(42)noise_levels = np.linspace(0, 1, 20)dice_losses_list = []ce_losses_list = []for noise in noise_levels:    pred = train_masks[0].copy()    n_corrupt = int(64 * 64 * noise)    corrupt_idx = np.random.choice(64 * 64, n_corrupt, replace=False)    pred.flat[corrupt_idx] = np.random.randint(0, 4, n_corrupt)        dl = dice_loss(pred, train_masks[0], 4)    ce = 1 - (pred == train_masks[0]).mean()    dice_losses_list.append(dl)    ce_losses_list.append(ce)fig, ax = plt.subplots(figsize=(10, 6))ax.plot(noise_levels, dice_losses_list, 'o-', linewidth=2, label='Dice Loss')ax.plot(noise_levels, ce_losses_list, 's-', linewidth=2, label='1 - Pixel Accuracy')ax.set_xlabel('Noise Level (fraction corrupted)')ax.set_ylabel('Loss')ax.set_title('Loss Functions vs Prediction Quality', fontsize=14, fontweight='bold')ax.legend(fontsize=11)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-14-segmentation/loss_comparison.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 10 saved: loss_comparison.png")

## 11.6 数据增强在分割中的应用### 直觉分割任务的数据增强需要同时对图像和mask做相同的变换。### 常见增强方法1. **水平翻转**：图像和mask同时翻转2. **随机裁剪**：在图像和mask的相同位置裁剪3. **弹性变形**：对医学图像特别有效4. **颜色抖动**：只对图像，不影响mask5. **旋转**：图像和mask同时旋转### 注意事项- 几何变换必须同时应用于图像和mask- 颜色变换只应用于图像- 使用插值时，mask应使用最近邻插值（避免产生新类别值）

In [29]:
# ============================================================# Data augmentation for segmentation# ============================================================def random_flip(img, mask, p=0.5):    if np.random.rand() < p:        img = img[:, ::-1, :].copy()        mask = mask[:, ::-1].copy()    return img, maskdef random_crop(img, mask, crop_size=48):    H, W = img.shape[:2]    i = np.random.randint(0, H - crop_size)    j = np.random.randint(0, W - crop_size)    return img[i:i+crop_size, j:j+crop_size], mask[i:i+crop_size, j:j+crop_size]def random_rotation(img, mask, max_angle=30):    # Simple rotation using array manipulation    angle = np.random.randint(-max_angle, max_angle)    # For simplicity, just do 90-degree rotations    k = np.random.randint(0, 4)  # 0, 90, 180, 270    img = np.rot90(img, k).copy()    mask = np.rot90(mask, k).copy()    return img, mask# Demonstrate augmentationsfig, axes = plt.subplots(2, 4, figsize=(16, 8))np.random.seed(42)aug_names = ['Original', 'Flip', 'Rotate 90', 'Crop 48']for col in range(4):    img_aug, mask_aug = train_images[0].copy(), train_masks[0].copy()    if col == 1:        img_aug, mask_aug = random_flip(img_aug, mask_aug)    elif col == 2:        img_aug, mask_aug = random_rotation(img_aug, mask_aug)    elif col == 3:        img_aug, mask_aug = random_crop(img_aug, mask_aug)        axes[0, col].imshow(img_aug)    axes[0, col].set_title(aug_names[col], fontsize=12)    axes[0, col].axis('off')    axes[1, col].imshow(mask_aug, cmap='nipy_spectral')    axes[1, col].axis('off')plt.suptitle('Data Augmentation for Segmentation', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-14-segmentation/augmentation.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 11 saved: augmentation.png")

## 11.7 分割模型的感受野分析### 直觉感受野决定了网络能"看到"的图像范围。对分割而言：- 小物体需要小感受野- 大物体需要大感受野- 最优策略：多尺度感受野### 手动计算感受野对于3x3卷积（stride=1），每层感受野增加2：| 层 | 操作 | 感受野 ||---|---|---|| 0 | 输入 | 1x1 || 1 | 3x3 conv | 3x3 || 2 | 3x3 conv | 5x5 || 3 | 3x3 conv + pool(2) | 7x7 (步长2) || 4 | 3x3 conv | 11x11 || 5 | 3x3 conv + pool(2) | 15x15 (步长4) |加入空洞卷积(dilation=2)后，同样3x3卷积感受野增加4而非2。

In [31]:
# ============================================================# Receptive field calculation# ============================================================def compute_receptive_field(layers):    # Compute receptive field given layer specs    rf = 1    jump = 1    for layer in layers:        if layer['type'] == 'conv':            rf = rf + (layer['kernel'] - 1) * jump * layer.get('dilation', 1)        elif layer['type'] == 'pool':            jump *= layer['stride']            rf = rf + (layer['kernel'] - 1) * jump        elif layer['type'] == 'dilated':            rf = rf + (layer['kernel'] - 1) * jump * layer['dilation']    return rf, jump# Compare architecturesarch_standard = [    {'type': 'conv', 'kernel': 3}, {'type': 'conv', 'kernel': 3},    {'type': 'pool', 'kernel': 2, 'stride': 2},    {'type': 'conv', 'kernel': 3}, {'type': 'conv', 'kernel': 3},    {'type': 'pool', 'kernel': 2, 'stride': 2},    {'type': 'conv', 'kernel': 3}, {'type': 'conv', 'kernel': 3},]arch_dilated = [    {'type': 'conv', 'kernel': 3}, {'type': 'conv', 'kernel': 3},    {'type': 'pool', 'kernel': 2, 'stride': 2},    {'type': 'dilated', 'kernel': 3, 'dilation': 2},    {'type': 'dilated', 'kernel': 3, 'dilation': 4},    {'type': 'dilated', 'kernel': 3, 'dilation': 8},]rf_std, jump_std = compute_receptive_field(arch_standard)rf_dil, jump_dil = compute_receptive_field(arch_dilated)print("=== Receptive Field Comparison ===")print(f"Standard CNN: RF={rf_std}x{rf_std}, jump={jump_std}")print(f"Dilated CNN:  RF={rf_dil}x{rf_dil}, jump={jump_dil}")print(f"Dilated has {rf_dil/rf_std:.1f}x larger RF at same resolution!")# Visualizefig, ax = plt.subplots(figsize=(10, 5))archs = ['Standard CNN\n(pool after conv2)', 'Dilated CNN\n(dilation 2,4,8)']rfs = [rf_std, rf_dil]ax.bar(archs, rfs, color=['steelblue', 'coral'], edgecolor='black', alpha=0.8)ax.set_ylabel('Receptive Field (pixels)')ax.set_title('Receptive Field: Standard vs Dilated', fontsize=14, fontweight='bold')for i, (a, r) in enumerate(zip(archs, rfs)):    ax.text(i, r + 1, f'{r}x{r}', ha='center', fontsize=12, fontweight='bold')ax.grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-14-segmentation/receptive_field.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 12 saved: receptive_field.png")

## 12. 关键要点总结| 概念 | 核心思想 ||---|---|| 语义分割 | 每像素一个类别 || 实例分割 | 每实例一个mask || FCN | 全卷积端到端 || U-Net | 对称编解码+跳跃 || 转置卷积 | 学习上采样 || 空洞卷积 | 不降分辨率的感受野扩大 || Skip连接 | 融合细节和语义 || mIoU | 标准评价指标 || ASPP | 多尺度特征 |**最佳实践：**1. 使用U-Net作为baseline2. 添加空洞卷积增大感受野3. 使用跳跃连接4. 后处理：CRF或形态学5. 评价指标用mIoU6. 数据增强：翻转、旋转、弹性变形

## 13. 作业### 作业1：实现FCN-8s在FCN-32s基础上添加跳跃连接：**要求：**1. 实现FCN-32s（单次上采样）2. 添加1/16特征融合 -> FCN-16s3. 添加1/8特征融合 -> FCN-8s4. 对比三者的mIoU5. 分析跳跃连接对边界精度的影响

### 作业2：实现DeepLab v3的ASPP模块ASPP使用多个不同空洞率的卷积并行处理：**要求：**1. 实现ASPP模块（rate=1, 6, 12, 18）2. 将ASPP接入分割网络3. 对比有无ASPP的效果4. 分析不同空洞率的作用5. 测试不同ASPP配置

### 作业3：实现边界损失 (Boundary Loss)边界损失关注分割边界质量：L_boundary = distance_transform(pred != gt)**要求：**1. 实现基于距离变换的边界损失2. 与标准交叉熵结合3. 对比不同权重(0.1, 0.5, 0.9)4. 分析边界精度提升

## 14. 参考文献1. [[Long et al., 2015]](https://arxiv.org/abs/1411.4038) - Fully Convolutional Networks for Semantic Segmentation (FCN)2. [[Ronneberger et al., 2015]](https://arxiv.org/abs/1505.04597) - U-Net: Convolutional Networks for Biomedical Image Segmentation3. [[Chen et al., 2017]](https://arxiv.org/abs/1606.00915) - DeepLab: Semantic Image Segmentation with Deep Convolutional Nets, Atrous Convolution4. [[Chen et al., 2018]](https://arxiv.org/abs/1802.02611) - Encoder-Decoder with Atrous Separable Convolution for Semantic Segmentation (DeepLab v3+)5. [[He et al., 2017]](https://arxiv.org/abs/1703.06870) - Mask R-CNN6. [[Zhao et al., 2017]](https://arxiv.org/abs/1612.01105) - Pyramid Scene Parsing Network (PSPNet)7. [[Xie et al., 2021]](https://arxiv.org/abs/2012.15840) - SegFormer: Simple and Efficient Design for Semantic Segmentation with Transformers8. [[Kervadec et al., 2019]](https://arxiv.org/abs/1812.07036) - Boundary loss for highly unbalanced segmentation9. [[Krähenbühl and Koltun, 2011]](https://arxiv.org/abs/1210.5644) - Efficient Inference in Fully Connected CRFs with Gaussian Edge Potentials10. [[CS231n, Stanford]](https://cs231n.github.io/) - CS231n: Deep Learning for Computer Vision